# 🛡️ Fine-Tuning do Laya — Detecção de Golpe do Pix

Treino do **Laya** (System 1, motor de decisão tipado não-autoregressivo) no domínio `pix_golpe`,
usando o dataset gerado localmente por `src/preparar_dataset.py`:

- **800 mensagens de treino** (`laya_pix_golpe_treino.jsonl`) — 60 golpes / 740 legítimas
- **200 mensagens de validação** (`laya_pix_golpe_validacao.jsonl`) — 15 golpes / 185 legítimas
- **4 perguntas tipadas** por mensagem (idênticas ao experimento original do `pix-golpe`):
  `golpe` (noul), `tipo` (choice, 8 classes), `urgencia` (score 0–2), `pede_pix` (noul)
- **Checkpoint base**: `convaiinnovations/laya` → subpasta `multilingual` (mmBERT-base, 322M, PT-BR)
- **Método**: RLCD — policy gradient (GRPO) com recompensa de scoring rule própria + cross-entropy,
  seguido de calibração de temperatura por tipo de pergunta

> ⚙️ **Antes de executar**: menu *Runtime → Change runtime type → T4 GPU*.
> Adaptado do notebook oficial de fine-tuning do Laya (Kaggle 2×T4) para **1× T4** do Colab.

## 1️⃣ Verificação da GPU

In [ ]:
!nvidia-smi

import torch

assert torch.cuda.is_available(), "❌ GPU não detectada. Ative: Runtime → Change runtime type → T4 GPU"
print(f"✅ GPU disponível: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")

## 2️⃣ Instalação das dependências

Instala o SDK do Laya e a stack de treino. Leva ~2 minutos no Colab.

In [ ]:
!pip install -q -U "laya>=0.3.21" "transformers>=5.0.0" safetensors huggingface_hub accelerate tabulate

import laya
import torch
import transformers

print("🧠 Laya        :", laya.__version__)
print("🤗 Transformers:", transformers.__version__)
print("🔥 PyTorch     :", torch.__version__)

## 3️⃣ Upload do dataset

Selecione os dois arquivos gerados localmente em `data/processed/`:

- `laya_pix_golpe_treino.jsonl`
- `laya_pix_golpe_validacao.jsonl`

In [ ]:
import json
from pathlib import Path

ARQ_TREINO = "laya_pix_golpe_treino.jsonl"
ARQ_VALIDACAO = "laya_pix_golpe_validacao.jsonl"

faltando = [a for a in (ARQ_TREINO, ARQ_VALIDACAO) if not Path(a).exists()]
if faltando:
    from google.colab import files

    print("📤 Selecione para upload:", ", ".join(faltando))
    files.upload()

for arq in (ARQ_TREINO, ARQ_VALIDACAO):
    linhas = [json.loads(linha) for linha in Path(arq).read_text(encoding="utf-8").splitlines() if linha.strip()]
    golpes = sum(1 for r in linhas if json.loads(r["gold"])["golpe"]["label"] == "true")
    print(f"📥 {arq}: {len(linhas)} mensagens ({golpes} golpes)")

## 4️⃣ Pré-processamento — sequências de treino

Cada mensagem vira **4 sequências** (uma por pergunta tipada) já tokenizadas com os marcadores
de opção do Laya: 800 × 4 = **3.200 itens de treino**.

In [ ]:
import os
from typing import Any

from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.common import QTYPES, build_sequence, render_options
from transformers import AutoTokenizer

MODEL_ID = "convaiinnovations/laya"
SUBPASTA = "multilingual"

base_dir = snapshot_download(MODEL_ID)
model_dir = os.path.join(base_dir, SUBPASTA)
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg: dict[str, Any] = json.load(f)
print("✅ Checkpoint multilingual | max_len:", cfg["max_len"], "| head_max_len:", cfg["head_max_len"])


def build_training_item(state: dict[str, Any], q: dict[str, Any], gold_q: dict[str, Any]) -> dict[str, Any] | None:
    """Tokeniza (state, pergunta, opções) e anexa o alvo calibrado do professor."""
    t = q["type"]
    crit = q.get("criteria", {})
    if t == "choice":
        keys = list(crit.keys())
        target = [gold_q["probabilities"].get(k, 0.0) for k in keys]
    elif t == "noul":
        target = [gold_q["probabilities"].get("false", 0.5), gold_q["probabilities"].get("true", 0.5)]
    else:  # score
        target = [gold_q["probabilities"].get(str(i), 0.0) for i in range(len(crit))]

    soma = sum(target)
    target = [v / soma for v in target] if soma > 0 else [1.0 / len(target)] * len(target)
    label = target.index(max(target))
    k = len(render_options({"t": t, "crit": crit}))

    seq, markers = build_sequence(
        tok, state, {"t": t, "ins": q["instructions"], "crit": crit}, cfg["max_len"], cfg["head_max_len"]
    )
    if len(markers) != k:
        return None
    return {"ids": seq, "markers": markers, "qtype": QTYPES[t], "target": target, "label": label}


linhas_treino = [json.loads(linha) for linha in Path(ARQ_TREINO).read_text(encoding="utf-8").splitlines() if linha.strip()]
items: list[dict[str, Any]] = []
descartados = 0
for row in linhas_treino:
    state = json.loads(row["state"])
    questions = json.loads(row["questions"])
    gold = json.loads(row["gold"])
    for qid, q in questions.items():
        item = build_training_item(state, q, gold[qid])
        if item is None:
            descartados += 1
            continue
        items.append(item)

torch.save(items, "/content/train_items.pt")
print(f"✅ {len(items)} sequências salvas em /content/train_items.pt (descartadas: {descartados})")

## 5️⃣ Treinamento — loop RLCD em 1× T4

Adaptação do `train_ddp.py` oficial (2×T4) para **GPU única**: `GRAD_ACCUM=8` mantém o batch
efetivo em 64 sequências. Um slice de calibração (10%, máx. 400 itens) é retirado do treino
**antes** do embaralhamento — idêntico ao protocolo oficial, para temperaturas honestas.

Hiperparâmetros (mesmos do notebook oficial):
| Parâmetro | Valor |
|---|---|
| Épocas | 4 |
| Micro-batch / acúmulo | 8 × 8 = 64 efetivo |
| LR encoder / head | 2.5e-5 / 1.0e-4 |
| Ruído de exploração σ | 0.4 → 0.1 (linear por época) |
| Recompensa | `w_sph=0.75`, `w_rps=1.0` + CE com peso 1.0 |

In [ ]:
%%writefile /content/train_laya.py
"""Treino single-GPU (Colab T4) do Laya no domínio pix_golpe — loop RLCD + calibração."""

import json
import os
import random
import sys
import time
from typing import Any

import torch
from laya.common import build_model, proper_reward
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer


def collate_train_batch(items: list[dict[str, Any]], pad_id: int) -> dict[str, torch.Tensor]:
    """Empilha sequências de tamanhos variáveis com padding e máscaras de opção."""
    n = len(items)
    comp = max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, comp), pad_id, dtype=torch.long)
    att = torch.zeros((n, comp), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids,
        "attention_mask": att,
        "marker_pos": mpos,
        "marker_mask": mmask,
        "target": target,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items]),
    }


def fit_one_temp(sel: list[tuple[Any, Any]]) -> float:
    """Ajusta uma temperatura de calibração (LBFGS) sobre logits e alvos retidos."""
    if len(sel) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    logits_z = torch.full((len(sel), kmax), -1e4)
    alvos = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        logits_z[i, : len(z)] = torch.tensor(z)
        alvos[i, : len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)

    def closure() -> torch.Tensor:
        opt.zero_grad()
        loss = -(alvos * torch.log_softmax(logits_z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss

    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())


def main() -> None:
    """Treina o checkpoint multilingual e salva o modelo calibrado em saída."""
    device = torch.device("cuda")
    model_dir, output_dir = sys.argv[1], sys.argv[2]

    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg: dict[str, Any] = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    model.load_state_dict(load_file(os.path.join(model_dir, "model.safetensors")), strict=True)
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    all_items: list[dict[str, Any]] = torch.load("/content/train_items.pt", weights_only=False)

    # Slice de calibração retido ANTES do treino (semente fixa = reprodutível).
    CALIB_MAX = 400
    order = list(range(len(all_items)))
    random.Random(20260922).shuffle(order)
    n_calib = min(CALIB_MAX, len(all_items) // 10)
    calib_items = [all_items[i] for i in sorted(order[:n_calib])]
    train_items = [all_items[i] for i in sorted(order[n_calib:])]

    EPOCHS = 4
    MICRO_BATCH = 8
    GRAD_ACCUM = 8  # batch efetivo = 64 sequências (8 x 8) em 1 GPU
    GROUP_SIZE = 4
    LR_ENCODER = 2.5e-5
    LR_HEAD = 1.0e-4
    SIGMA_START = 0.4
    SIGMA_END = 0.1

    enc_params = [p for n, p in model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in model.named_parameters() if "encoder." not in n]
    optimizer = torch.optim.AdamW(
        [{"params": enc_params, "lr": LR_ENCODER}, {"params": head_params, "lr": LR_HEAD}],
        weight_decay=0.01,
    )
    total_updates = (len(train_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    print(
        f"🚀 Treino single-GPU: {len(train_items)} itens (+{len(calib_items)} calibração) | {EPOCHS} épocas",
        flush=True,
    )
    t0 = time.time()

    for epoch in range(EPOCHS):
        random.seed(42 + epoch)
        random.shuffle(train_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0

        progresso = epoch / max(1, EPOCHS - 1)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * progresso

        for b_idx in range(0, len(train_items), MICRO_BATCH):
            chunk = train_items[b_idx : b_idx + MICRO_BATCH]
            if not chunk:
                continue
            batch = collate_train_batch(chunk, tok.pad_token_id)

            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device),
                )
            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(device)

            # 1. Amostra G perturbações de logits com projeção de soma zero.
            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)

            # 2. Recompensa de scoring rule própria + vantagem baselined.
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask, w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)

            # 3. Policy gradient + cross-entropy suave (peso 1.0).
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma**2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM + 0.0 * act.sum()

            scaler.scale(loss).backward()
            accum_step += 1

            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(train_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1
            if n_batches % 50 == 0:
                print(
                    f"  Época {epoch + 1}/{EPOCHS} | step {n_batches} | "
                    f"loss {loss.item() * GRAD_ACCUM:.4f} | reward {r.mean().item():.3f} | "
                    f"lr {scheduler.get_last_lr()[0]:.2e}",
                    flush=True,
                )

        loss_medio = epoch_loss / max(1, n_batches)
        print(f"=== Época {epoch + 1}/{EPOCHS} | {time.time() - t0:.1f}s | loss médio {loss_medio:.4f} ===", flush=True)

        # Checkpoint rolante por época: proteção contra timeout/OOM da sessão Colab.
        ckpt_dir = os.path.join(output_dir, "checkpoint_latest")
        os.makedirs(ckpt_dir, exist_ok=True)
        save_file(
            {k_: v.half().contiguous().cpu() for k_, v in model.state_dict().items()},
            os.path.join(ckpt_dir, "model.safetensors"),
        )
        model.encoder.config.save_pretrained(os.path.join(ckpt_dir, "encoder"))
        tok.save_pretrained(os.path.join(ckpt_dir, "tokenizer"))
        with open(os.path.join(ckpt_dir, "checkpoint_meta.json"), "w") as f:
            meta = {"epoch": epoch + 1, "total_epochs": EPOCHS, "avg_loss": epoch_loss / max(1, n_batches)}
            json.dump(meta, f, indent=2)
        print(f"  💾 Checkpoint rolante salvo (época {epoch + 1}/{EPOCHS})", flush=True)

    # Calibração de temperatura pós-treino sobre o slice retido (lotes de 16).
    print("\n🌡️ Ajustando temperaturas de calibração...", flush=True)
    del optimizer, scaler, scheduler
    torch.cuda.empty_cache()
    model.eval()
    calib_preds: list[tuple[int, Any, Any]] = []
    with torch.no_grad():
        for c_idx in range(0, len(calib_items), 16):
            c_chunk = calib_items[c_idx : c_idx + 16]
            cb = collate_train_batch(c_chunk, tok.pad_token_id)
            with torch.autocast("cuda", dtype=torch.float16):
                l_sub, _ = model(
                    cb["input_ids"].to(device),
                    cb["attention_mask"].to(device),
                    cb["marker_pos"].to(device),
                    cb["marker_mask"].to(device),
                    cb["qtype"].to(device),
                )
            l_np = l_sub.float().cpu().numpy()
            for r_, it in enumerate(c_chunk):
                calib_preds.append((it["qtype"], l_np[r_, : len(it["markers"])], it["target"]))

    fitted_temps = [1.2, 1.2, 1.2]
    try:
        for qt in range(3):
            sel = [(z, t) for q_type, z, t in calib_preds if q_type == qt]
            if sel:
                fitted_temps[qt] = fit_one_temp(sel)
        print("🌡️ Temperaturas (choice, score, noul):", [round(t, 3) for t in fitted_temps], flush=True)
    except Exception as exc:  # fallback conservador do notebook oficial
        print("⚠️ Calibração falhou, usando 1.2:", exc, flush=True)

    os.makedirs(output_dir, exist_ok=True)
    save_file(
        {k_: v.half().contiguous().cpu() for k_, v in model.state_dict().items()},
        os.path.join(output_dir, "model.safetensors"),
    )
    model.encoder.config.save_pretrained(os.path.join(output_dir, "encoder"))
    tok.save_pretrained(os.path.join(output_dir, "tokenizer"))

    cfg["fine_tuned"] = True
    cfg["model_name"] = "laya-pix-golpe"
    cfg["temperature"] = fitted_temps
    cfg.pop("temperature_by_options", None)
    with open(os.path.join(output_dir, "rl_agent_config.json"), "w") as f:
        json.dump(cfg, f, indent=2)
    print(f"✅ Modelo fine-tunado salvo em {output_dir}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
OUT_DIR = "/content/laya_pix_golpe_finetuned"

!python /content/train_laya.py "{model_dir}" "{OUT_DIR}"


## 6️⃣ Avaliação no split de validação

Avalia as 4 perguntas tipadas nas **200 mensagens de validação** e aplica o protocolo de
veredito em 3 faixas do experimento original (`golpe ≥ 0,6` → GOLPE; `≤ 0,4` → OK; caso
contrário → SUSPEITA), com acurácia, precisão e recall idênticos ao `paper/` do repositório.

In [ ]:
import time

import numpy as np
import laya
from laya.common import ece_score

agent = laya.Agent(OUT_DIR, device="cuda")
linhas_val = [json.loads(linha) for linha in Path(ARQ_VALIDACAO).read_text(encoding="utf-8").splitlines() if linha.strip()]

resultados: list[dict[str, Any]] = []
latencias: list[float] = []
for row in linhas_val:
    state = json.loads(row["state"])
    questions = json.loads(row["questions"])
    gold = json.loads(row["gold"])
    t0 = time.perf_counter()
    res = agent.predict(state, questions)
    latencias.append((time.perf_counter() - t0) * 1000)
    resultados.append({"gold": gold, "pred": res["answers"]})


def veredito(p_golpe: float) -> str:
    """Veredito em 3 faixas do experimento pix-golpe."""
    return "golpe" if p_golpe >= 0.6 else "ok" if p_golpe <= 0.4 else "revisar"


acertos: dict[str, list[float]] = {qid: [] for qid in ("golpe", "tipo", "urgencia", "pede_pix")}
confs: list[float] = []
corretos: list[float] = []
matriz: dict[str, dict[str, int]] = {real: {pred: 0 for pred in ("golpe", "ok", "revisar")} for real in ("golpe", "normal")}
mae_urgencia: list[float] = []

for item in resultados:
    gold, pred = item["gold"], item["pred"]

    # golpe (noul)
    p = float(pred["golpe"]["noul"])
    acertos["golpe"].append(float(("true" if p >= 0.5 else "false") == gold["golpe"]["label"]))
    confs.append(max(p, 1.0 - p))
    corretos.append(acertos["golpe"][-1])

    # tipo (choice, 8 classes)
    ok_tipo = float(pred["tipo"]["choice"] == gold["tipo"]["label"])
    acertos["tipo"].append(ok_tipo)
    confs.append(float(pred["tipo"]["probabilities"][pred["tipo"]["choice"]]))
    corretos.append(ok_tipo)

    # urgencia (score 0-2)
    probs_u = np.array([pred["urgencia"]["probabilities"].get(str(i), 0.0) for i in range(3)])
    nivel_pred = int(np.argmax(probs_u)) if probs_u.sum() > 0 else int(round(float(pred["urgencia"]["score"])))
    ok_urg = float(nivel_pred == int(gold["urgencia"]["label"]))
    acertos["urgencia"].append(ok_urg)
    mae_urgencia.append(abs(float(pred["urgencia"]["score"]) - float(gold["urgencia"]["label"])))
    confs.append(float(probs_u.max()) if probs_u.sum() > 0 else 0.5)
    corretos.append(ok_urg)

    # pede_pix (noul)
    p_pix = float(pred["pede_pix"]["noul"])
    acertos["pede_pix"].append(float(("true" if p_pix >= 0.5 else "false") == gold["pede_pix"]["label"]))
    confs.append(max(p_pix, 1.0 - p_pix))
    corretos.append(acertos["pede_pix"][-1])

    # veredito 3 faixas vs verdade de referência
    real = "golpe" if gold["golpe"]["label"] == "true" else "normal"
    matriz[real][veredito(p)] += 1

tp = matriz["golpe"]["golpe"]
fn = matriz["golpe"]["ok"] + matriz["golpe"]["revisar"]
fp = matriz["normal"]["golpe"]
tn = matriz["normal"]["ok"] + matriz["normal"]["revisar"]

print("=" * 60)
print("📐 ACURÁCIA POR PERGUNTA TIPADA")
for qid, lista in acertos.items():
    print(f"   {qid:<10}: {np.mean(lista):.3f}")
print(f"   {'GLOBAL':<10}: {np.mean(corretos):.3f}  ({len(corretos)} decisões)")
print(f"   ECE            : {ece_score(np.array(confs), np.array(corretos)):.3f}")
print(f"   MAE urgencia   : {np.mean(mae_urgencia):.3f}")
print(f"   latência p50/p95: {np.percentile(latencias, 50):.0f} ms / {np.percentile(latencias, 95):.0f} ms")
print("=" * 60)
print("🧭 VEREDITO 3 FAIXAS (protocolo do paper)")
print(f"   real\\pred   golpe   ok   revisar")
for real in ("golpe", "normal"):
    print(f"   {real:<9} {matriz[real]['golpe']:>6} {matriz[real]['ok']:>5} {matriz[real]['revisar']:>8}")
total = tp + fn + fp + tn
print(f"   acurácia: {(tp + tn) / total:.3f} | precisão: {tp / max(1, tp + fp):.3f} | recall: {tp / max(1, tp + fn):.3f}")
print("   Referência do teacher (DeepSeek) no dataset completo: acurácia 1.000")

## 7️⃣ Exportação ONNX INT8 (para CPU local)

O notebook local tem RAM limitada, então exportamos o modelo para **ONNX com quantização
dinâmica INT8 por canal** (~4× menor que o checkpoint fp16, roda com `onnxruntime` sem
carregar o PyTorch). Usa o `scripts/export_onnx.py` oficial do repositório do Laya.

In [ ]:
!curl -sL "https://raw.githubusercontent.com/NandhaKishorM/laya/main/scripts/export_onnx.py" -o /content/export_onnx.py
!pip install -q onnx onnxruntime onnxscript

# Exporta fp32 temporário + cópia INT8 per-channel; só o INT8 entra no pacote final
!python /content/export_onnx.py --model "{OUT_DIR}" --output "/content/laya_fp32.onnx" --quantize

import shutil

shutil.move("/content/laya_fp32.int8.onnx", f"{OUT_DIR}/laya.int8.onnx")
!rm -f /content/laya_fp32.onnx
print("🧊 ONNX INT8 gravado em:", f"{OUT_DIR}/laya.int8.onnx")

## 8️⃣ Download do modelo

Gera um `.zip` do checkpoint fine-tunado (torch + ONNX INT8) e baixa para a sua máquina.
Descompacte em `models/laya_pix_golpe_finetuned/` no projeto local.

In [ ]:
import shutil

from google.colab import files

zip_path = shutil.make_archive("/content/laya_pix_golpe_finetuned", "zip", OUT_DIR)
print("📦 Pacote gerado:", zip_path)
files.download(zip_path)
print("✅ Download iniciado — descompacte o CONTEÚDO em models/laya_pix_golpe_finetuned/ no projeto local")

---
📚 **Proveniência**: dataset `sandeco/pix-golpe` (1000 mensagens sintéticas, semente 42) ×
decisões do teacher DeepSeek (`paper/results/.../decisions.csv`, acurácia 1.000) ·
perguntas tipadas de `src/jev.rs` · protocolo de treino do notebook oficial
`laya_finetune_typed_decisions` (RLCD, 2×T4 → adaptado para 1×T4) · exportação INT8 via
`scripts/export_onnx.py` oficial para inferência local em CPU (`src/servidor_web.py`).